## EDA - Price Cannibalization Project
##### Dataset: transaction_1.csv + transaction_2.csv

In [1]:
# import

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

In [6]:
# config

import os 

DATA_DIR = "dataset/"      
OUTPUT_DIR = "eda_output/"        

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(os.getcwd())
print(os.listdir())
print(os.path.exists(DATA_DIR))

d:\Intern - Capstone Project\eda
['dataset', 'eda_output', 'notebook.ipynb']
True


In [7]:
# config

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "grid.linestyle":   "--",
    "font.size":        11,
})

BRAND_COLOR = {
    "Richeese": "#378ADD",
    "Richoco":  "#D4537E",
    "Nextar":   "#1D9E75",
}
CAT_COLOR = {
    "Wafer":          "#378ADD",
    "Kue/Pie":        "#1D9E75",
    "Minuman":        "#D4537E",
    "Mi Instan":      "#BA7517",
    "Extruded Snack": "#534AB7",
    "Biskuit":        "#888780",
}

In [10]:
# load and basic prep

print("=" * 60)
print("STEP 1 - Load & Basic Preparation")
print("=" * 60)

df1 = pd.read_csv(DATA_DIR + "transaction_1.csv", index_col=0)
df2 = pd.read_csv(DATA_DIR + "transaction_2.csv", index_col=0)
df  = pd.concat([df1, df2], ignore_index=True)

df["DateTime"] = pd.to_datetime(df["DateTime"])
df["Date"]     = df["DateTime"].dt.date
df["Week"]     = df["DateTime"].dt.to_period("W").apply(lambda r: r.start_time.date())
df["Month"]    = df["DateTime"].dt.to_period("M")

print(f"Total rows    : {len(df):,}")
print(f"Columns       : {list(df.columns)}")
print(f"Date range    : {df['DateTime'].min().date()} → {df['DateTime'].max().date()}")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nBrands     : {sorted(df['Brand'].unique())}")
print(f"Categories : {sorted(df['SKU_Category'].unique())}")
print(f"Promo types: {sorted(df['Promo_Category'].unique())}")

STEP 1 - Load & Basic Preparation
Total rows    : 500,002
Columns       : ['TransactionID', 'DateTime', 'Date_Key', 'Branch', 'Customer_ID', 'SKU_ID', 'SKU', 'Brand', 'SKU_Category', 'Qty', 'NormalPrice', 'DiscountedPrice', 'DiscountPercentage', 'Total_Price', 'Promo_Category', 'Date', 'Week', 'Month']
Date range    : 2025-05-01 → 2025-10-30
Missing values:
TransactionID         0
DateTime              0
Date_Key              0
Branch                0
Customer_ID           0
SKU_ID                0
SKU                   0
Brand                 0
SKU_Category          0
Qty                   0
NormalPrice           0
DiscountedPrice       0
DiscountPercentage    0
Total_Price           0
Promo_Category        0
Date                  0
Week                  0
Month                 0
dtype: int64

Brands     : ['Nextar', 'Richeese', 'Richoco']
Categories : ['Biskuit', 'Extruded Snack', 'Kue/Pie', 'Mi Instan', 'Minuman', 'Wafer']
Promo types: ['No Promo', 'Promo Bundling', 'Promo Normal', 

In [11]:
print(df1['DateTime'].min(), df1['DateTime'].max())
print(df2['DateTime'].min(), df2['DateTime'].max())

2025-05-01 09:01:41 2025-10-30 20:56:19
2025-05-01 09:00:21 2025-10-30 20:56:20


In [14]:
# sku summary

print("\n" + "=" * 60)
print("STEP 2 - SKU Summary")
print("=" * 60)

sku_summary = (
    df.groupby(["SKU_ID", "SKU", "Brand", "SKU_Category"])
    .agg(
        total_qty       = ("Qty", "sum"),
        total_revenue   = ("Total_Price", "sum"),
        transactions    = ("TransactionID", "nunique"),
        normal_price    = ("NormalPrice", "median"),
        avg_discount_pct= ("DiscountPercentage", "mean"),
        max_discount_pct= ("DiscountPercentage", "max"),
    )
    .reset_index()
    .sort_values("SKU_ID")
)
sku_summary["avg_discount_pct_pct"] = sku_summary["avg_discount_pct"] * 100

print(sku_summary[["SKU_ID", "SKU", "Brand", "SKU_Category",
                    "total_qty", "normal_price", "avg_discount_pct_pct"]].to_string(index=False))


STEP 2 - SKU Summary
SKU_ID                               SKU    Brand   SKU_Category  total_qty  normal_price  avg_discount_pct_pct
  S001           Richeese Wafer Keju 50g Richeese          Wafer     143659        5500.0              5.227116
  S002         Richoco Wafer Cokelat 50g  Richoco          Wafer      67266        6100.0              5.292990
  S003   Richeese Wafer Keju 10g Renceng Richeese          Wafer      67306       10000.0              5.225956
  S004 Richoco Wafer Cokelat 10g Renceng  Richoco          Wafer      67626       10000.0              5.266068
  S005           Nextar Brownies Pie 40g   Nextar        Kue/Pie      67487        8000.0              5.197181
  S006             Nextar Nastar Pie 30g   Nextar        Kue/Pie      66937        8000.0              5.208324
  S007            Richeese Siip Keju 20g Richeese Extruded Snack      68164        3000.0              5.276771
  S008         Richoco Ahh! Extruded 15g  Richoco Extruded Snack      66627       